In [4]:
import os
from pathlib import Path
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
)
from transformers import CLIPTokenizer, CLIPTextModel


2025-12-08 23:27:56.933750: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765236476.960403    5115 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765236476.972244    5115 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-08 23:27:57.178629: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
@dataclass
class TrainConfig:
    ROOT: str = "/root/wound_data" 
    OUTPUT_DIR: str = "./wg2diff_checkpoints"

    # training hyperparams
    image_size: int = 512
    train_batch_size: int = 4     
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-5
    num_train_steps: int = 20000    
    lr_warmup_steps: int = 500

    seed: int = 42
    mixed_precision: str = "fp16" 
    model_name: str = "runwayml/stable-diffusion-v1-5"
    checkpoint_every: int = 1000

config = TrainConfig()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
Path(config.OUTPUT_DIR).mkdir(exist_ok=True, parents=True)


Device: cuda


In [ ]:
class WoundMaskDataset(Dataset):
    def __init__(self, roots, tokenizer, image_size=512, prompt="a clinical wound photograph"):
        
        self.tokenizer = tokenizer
        self.image_size = image_size
        self.prompt = prompt

        self.items = []
        for img_dir, mask_dir in roots:
            img_dir = Path(img_dir)
            mask_dir = Path(mask_dir)
            for img_path in img_dir.glob("*"):
                if img_path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
                    continue
                mask_path = mask_dir / img_path.name
                if mask_path.exists():
                    self.items.append((img_path, mask_path))

        if not self.items:
            raise ValueError("No (image, mask) pairs found. Check your paths & filenames.")

        self.img_tf = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.mask_tf = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.NEAREST),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),   # 1×H×W, values ~0–1
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, mask_path = self.items[idx]
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")   # B/W mask

        img = self.img_tf(img)
        mask = self.mask_tf(mask)                  # [1,H,W]

        # binarize mask hard at 0.5
        mask = (mask > 0.5).float()

        tokens = self.tokenizer(
            self.prompt,
            truncation=True,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        )

        return {
            "pixel_values": img,
            "mask": mask,
            "input_ids": tokens.input_ids[0],
            "attention_mask": tokens.attention_mask[0],
        }
# build roots list using your folder layout
ROOT = Path("final_dataset/final_dataset")

roots = [
    (ROOT / "Kaggle_images", ROOT / "Kaggle_Masks"),
    (ROOT / "AZH",           ROOT / "AZH_masks"),
    (ROOT / "WoundDB",       ROOT / "WoundDB_masks"),
]

print("Roots:")
for img_dir, mask_dir in roots:
    print(" ", img_dir, "|", mask_dir)


Roots:
  final_dataset/final_dataset/Kaggle_images | final_dataset/final_dataset/Kaggle_Masks
  final_dataset/final_dataset/AZH | final_dataset/final_dataset/AZH_masks
  final_dataset/final_dataset/WoundDB | final_dataset/final_dataset/WoundDB_masks


In [10]:
# set seeds
random.seed(config.seed)
torch.manual_seed(config.seed)
if device.type == "cuda":
    torch.cuda.manual_seed_all(config.seed)

tokenizer = CLIPTokenizer.from_pretrained(config.model_name, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(config.model_name, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(config.model_name, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(config.model_name, subfolder="unet").to(device)
noise_scheduler = DDPMScheduler.from_pretrained(config.model_name, subfolder="scheduler")

# freeze VAE + text encoder
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(True)

# modify UNet to accept 4 latent channels + 1 mask channel = 5
old_conv = unet.conv_in
new_conv = nn.Conv2d(
    in_channels=old_conv.in_channels + 1,   # +1 for mask
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
)

with torch.no_grad():
    # copy original weights
    new_conv.weight[:, :old_conv.in_channels] = old_conv.weight
    # init mask channel with zeros
    new_conv.weight[:, old_conv.in_channels:] = 0.0
    new_conv.bias = old_conv.bias

unet.conv_in = new_conv

print("UNet input channels:", unet.conv_in.in_channels)  # should be 5 now


UNet input channels: 5


In [11]:
dataset = WoundMaskDataset(
    roots=roots,
    tokenizer=tokenizer,
    image_size=config.image_size,
    prompt="a high quality clinical photograph of a chronic wound",
)

print("Number of image/mask pairs:", len(dataset))

dataloader = DataLoader(
    dataset,
    batch_size=config.train_batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=config.learning_rate,
    betas=(0.9, 0.999),
    weight_decay=1e-2,
)

def get_lr(step):
    # linear warmup then cosine decay
    if step < config.lr_warmup_steps:
        return config.learning_rate * step / max(1, config.lr_warmup_steps)
    progress = (step - config.lr_warmup_steps) / max(1, config.num_train_steps - config.lr_warmup_steps)
    progress = min(max(progress, 0.0), 1.0)
    return config.learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and config.mixed_precision == "fp16"))


Number of image/mask pairs: 3769


/tmp/ipykernel_5115/2333674858.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and config.mixed_precision == "fp16"))


In [ ]:
import torch
------------------------------------------------------------
        # Encode image → latent space
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()     # [B,4,64,64]
            latents = latents * 0.18215                                # SD scaling

            # Downsample mask to latent resolution
            mask_latents = F.interpolate(
                masks,
                size=latents.shape[-2:], 
                mode="nearest",
            )                                                           # [B,1,64,64]

        # 2. Add noise to latents
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,   # NEW: use .config.
            (bsz,),
            device=device,
            dtype=torch.long,
        )

        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # 3. Text conditioning (no grad)
        with torch.no_grad():
            encoder_hidden_states = text_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            ).last_hidden_state

        # 4. Concatenate mask channel → final UNet input
        # noisy_latents: [B,4,H,W]
        # mask_latents: [B,1,H,W]
        unet_input = torch.cat([noisy_latents, mask_latents], dim=1)   # [B,5,64,64]
        # 5. Predict noise (FP32)
        model_pred = unet(
            unet_input,
            timesteps,
            encoder_hidden_states=encoder_hidden_states,
        ).sample

        loss = F.mse_loss(model_pred, noise)

        # 6. Optimizer step (FP32)
        # Learning-rate warmup + cosine decay
        lr = get_lr(global_step)
        for g in optimizer.param_groups:
            g["lr"] = lr

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        global_step += 1

        if global_step % print_every == 0:
            print(f"step {global_step}/{config.num_train_steps} | loss: {loss.item():.4f} | lr: {lr:.2e}")

        # 7. Save UNet checkpoint
        if global_step % config.checkpoint_every == 0:
            ckpt_dir = Path(config.OUTPUT_DIR) / f"step_{global_step}"
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            unet.save_pretrained(ckpt_dir)
            print(f"Saved checkpoint at {ckpt_dir}")

print("Training complete!")


Starting training...
step 50/20000 | loss: 0.1110 | lr: 9.80e-07
step 100/20000 | loss: 0.0548 | lr: 1.98e-06
step 150/20000 | loss: 0.3114 | lr: 2.98e-06
step 200/20000 | loss: 0.0133 | lr: 3.98e-06
step 250/20000 | loss: 0.1322 | lr: 4.98e-06
step 300/20000 | loss: 0.0944 | lr: 5.98e-06
step 350/20000 | loss: 0.0594 | lr: 6.98e-06
step 400/20000 | loss: 0.0253 | lr: 7.98e-06
step 450/20000 | loss: 0.0148 | lr: 8.98e-06
step 500/20000 | loss: 0.0244 | lr: 9.98e-06
step 550/20000 | loss: 0.0564 | lr: 1.00e-05
step 600/20000 | loss: 0.1229 | lr: 1.00e-05
step 650/20000 | loss: 0.0988 | lr: 1.00e-05
step 700/20000 | loss: 0.0377 | lr: 1.00e-05
step 750/20000 | loss: 0.1302 | lr: 1.00e-05
step 800/20000 | loss: 0.0091 | lr: 9.99e-06
step 850/20000 | loss: 0.2539 | lr: 9.99e-06
step 900/20000 | loss: 0.0602 | lr: 9.99e-06
step 950/20000 | loss: 0.2142 | lr: 9.99e-06
step 1000/20000 | loss: 0.3449 | lr: 9.98e-06
Saved checkpoint at wg2diff_checkpoints/step_1000
step 1050/20000 | loss: 0.043